# LeetCode 945 — Minimum Increment to Make Array Unique bằng Python

Link đề: https://leetcode.com/problems/minimum-increment-to-make-array-unique/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên; có thể lên tới `~5·10^9`, **vượt `int` 32-bit** nếu mọi phần tử cùng bằng `10^5`, nên ở ngôn ngữ khác cần `long`). Điểm cốt lõi: chỉ được **tăng** giá trị. Sau khi sắp xếp, đặt mỗi phần tử vào ô trống nhỏ nhất **không nhỏ hơn** nó và **lớn hơn** ô vừa dùng: `y = max(y + 1, x)`, cộng `y − x`. Oracle: **vét cạn mọi cách chọn tập ô đích phân biệt** và ghép tối ưu (kiểm tra thêm bằng hoán vị đầy đủ trên cỡ nhỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi nước tăng một phần tử thêm `1`; tìm số nước **ít nhất** để mọi giá trị **khác nhau**.
2. Nắm công cụ Python: `sorted`, mảng đếm (counting) với "mang số dư", cấu trúc **Union-Find** để tìm ô trống kế tiếp, `heapq`.
3. Đi từ **sắp xếp + tham lam `y = max(y + 1, x)`** → **mảng đếm và dồn trùng sang giá trị kế tiếp** → **Union-Find "ô trống kế tiếp"** → **điền các khe trống bằng hai con trỏ**.
4. Hiểu mấu chốt: tổng số nước `= Σ (giá trị đích) − Σ (giá trị gốc)`; để tổng đích nhỏ nhất, các phần tử (đã sắp xếp) nên đến các ô **liên tiếp tăng dần** tính từ giá trị của chúng; tham lam từ nhỏ đến lớn là tối ưu.
5. Tránh lỗi kinh điển: quên sắp xếp, chỉ đếm số phần tử trùng (mỗi trùng chỉ cần `+1`), dùng `y − x` mà không `max` (kết quả âm), khởi tạo `y` sai (`0` thay vì `-1`), và tràn `int` 32-bit khi cộng dồn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trong một nước đi, chọn chỉ số `i` bất kỳ và **tăng** `nums[i]` lên `1`.

Trả về số nước đi **tối thiểu** để mọi giá trị trong `nums` **khác nhau từng đôi**. (Test được sinh sao cho đáp án vừa trong số nguyên 32-bit.)

**Ví dụ 1:** `nums = [1,2,2]` → `1` (có thể thành `[1,2,3]`).

**Ví dụ 2:** `nums = [3,2,1,2,1,7]` → `6` (có thể thành `[3,4,1,2,5,7]`; không thể với `5` nước trở xuống).

**Ràng buộc:** `1 <= nums.length <= 10^5`; `0 <= nums[i] <= 10^5`.

In [1]:
import itertools
import random
import time
from heapq import heapify, heappop, heappush
from typing import List

examples = [([1, 2, 2], 1), ([3, 2, 1, 2, 1, 7], 6)]
for a, want in examples:
    print(f"nums = {a} -> {want}")

nums = [1, 2, 2] -> 1
nums = [3, 2, 1, 2, 1, 7] -> 6


## 3. Kiến thức nền

### 3.1 Đổi bài toán thành "gán ô đích"

Mỗi phần tử `x` được tăng thành `t ≥ x` (chi phí `t − x`); các `t` phải **phân biệt**. Tổng chi phí `= Σ t − Σ x`. Vậy cần chọn `n` giá trị đích phân biệt, ghép với các phần tử (mỗi `t ≥` phần tử được ghép) sao cho `Σ t` nhỏ nhất.

### 3.2 Tham lam sau khi sắp xếp

Sắp tăng `x₁ ≤ x₂ ≤ … ≤ x_n`. Gọi `t_i` là đích của `x_i`. Chọn `t_1 = x_1` và `t_i = max(x_i, t_{i−1} + 1)` — luôn chọn ô trống nhỏ nhất hợp lệ (không nhỏ hơn `x_i`, lớn hơn các đích đã dùng). Chi phí `Σ (t_i − x_i)`.

In [2]:
def greedy_targets(nums):
    xs = sorted(nums)
    t, out = -1, []
    for x in xs:
        t = max(t + 1, x)
        out.append(t)
    return xs, out


xs, ts = greedy_targets([3, 2, 1, 2, 1, 7])
print("sắp xếp:", xs, "| đích:", ts, "| chi phí:", sum(ts) - sum(xs))

sắp xếp: [1, 1, 2, 2, 3, 7] | đích: [1, 2, 3, 4, 5, 7] | chi phí: 6


### 3.3 Mảng đếm và "mang số dư"

Giá trị nằm trong `[0, 10^5]`. Đếm `cnt[v]`; duyệt `v` tăng dần, giữ `carry` = số phần tử "dư" cần đẩy sang giá trị lớn hơn: tại `v`, tổng `cnt[v] + carry` phần tử muốn ở `v`; một phần tử ở lại, `cnt[v] + carry − 1` đẩy tiếp (mỗi phần tử bị đẩy tốn `1` nước **cho mỗi bước**, tổng chi phí bước đó bằng số phần tử bị đẩy).

### 3.4 Chi phí `Σ t − Σ x` và tràn số

Trường hợp xấu nhất: `10^5` phần tử bằng `10^5` ⇒ đích `10^5, 10^5 + 1, …, 10^5 + 99999` ⇒ chi phí `0 + 1 + … + 99999 = 4 999 950 000 > 2^31 − 1`. Python không tràn; ở C++/Java dùng `long`.

In [3]:
n, v = 10 ** 5, 10 ** 5
cost = n * (n - 1) // 2
print("chi phí lớn nhất:", cost, "| > 2^31 - 1:", cost > 2 ** 31 - 1)

chi phí lớn nhất: 4999950000 | > 2^31 - 1: True


## 4. Ý tưởng nền tảng / trực giác

`nums = [3, 2, 1, 2, 1, 7]`, sắp xếp: `[1, 1, 2, 2, 3, 7]`.

| `x` | `t` trước | `t = max(t + 1, x)` | chi phí `t − x` |
|---|---|---|---|
| 1 | −1 | 1 | 0 |
| 1 | 1 | 2 | 1 |
| 2 | 2 | 3 | 1 |
| 2 | 3 | 4 | 2 |
| 3 | 4 | 5 | 2 |
| 7 | 5 | 7 | 0 |

Tổng `0 + 1 + 1 + 2 + 2 + 0 = 6` ✓. Phần tử `7` đã "thoát" khỏi chuỗi dồn vì `max(6, 7) = 7`.

In [4]:
xs = sorted([3, 2, 1, 2, 1, 7])
t, total = -1, 0
for x in xs:
    t = max(t + 1, x)
    total += t - x
    print(f"x={x}: đích {t}, chi phí {t - x}, tổng {total}")

x=1: đích 1, chi phí 0, tổng 0
x=1: đích 2, chi phí 1, tổng 1
x=2: đích 3, chi phí 1, tổng 2
x=2: đích 4, chi phí 2, tổng 4
x=3: đích 5, chi phí 2, tổng 6
x=7: đích 7, chi phí 0, tổng 6


## 5. Cách 1 — Sắp xếp + tham lam

**Ý tưởng:** sắp xếp; duyệt từ nhỏ đến lớn, giữ `t` = đích vừa dùng; đích mới là `max(t + 1, x)`; cộng `t − x`.

In [5]:
def increments_greedy(nums):
    t = -1                                               # giá trị đích đã dùng lớn nhất
    moves = 0
    for x in sorted(nums):
        t = max(t + 1, x)                                # ô trống nhỏ nhất không nhỏ hơn x và lớn hơn mọi đích trước
        moves += t - x
    return moves


for a, want in examples:
    assert increments_greedy(a) == want
print([increments_greedy(a) for a, _ in examples])

[1, 6]


In [6]:
# Code nộp LeetCode cho Cách 1
class SolutionGreedy:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        nums.sort()
        t, moves = -1, 0
        for x in nums:
            t = max(t + 1, x)
            moves += t - x
        return moves

**Độ phức tạp:** `O(n log n)` thời gian (sắp xếp), `O(1)` bộ nhớ thêm (hoặc `O(n)` nếu không sắp tại chỗ).

**Nút thắt:** sắp xếp `O(n log n)`; giá trị chỉ trong `[0, 10^5]` nên có thể đếm.

## 6. Ý tưởng thuật toán / bước đột phá

Có ba góc nhìn tương đương:

1. **Sắp xếp + tham lam** (Cách 1).
2. **Đếm + mang số dư:** duyệt giá trị `v` tăng dần; phần tử trùng ở `v` được đẩy sang `v + 1`, tốn `1` nước cho mỗi phần tử bị đẩy (Cách 2).
3. **Ô trống kế tiếp:** mỗi phần tử tìm ô trống nhỏ nhất `≥ x` bằng Union-Find (Cách 3).

## 7. Các cách tiếp theo

Ngoài Cách 1, ta có thêm 3 cách: **Cách 2** mảng đếm + mang số dư, **Cách 3** Union-Find "ô trống kế tiếp", **Cách 4** điền các khe trống bằng hai con trỏ.

### Cách 2 — Mảng đếm + mang số dư (counting)

**Giải thích từng đoạn:**

1. `cnt[v]` đếm số phần tử bằng `v`, với `v ∈ [0, max + n]`.
2. Duyệt `v` từ `0`: `carry` là số phần tử đang chờ "đẩy". Tại `v`: `total_here = cnt[v] + carry`. Nếu `total_here > 0` thì đặt **một** phần tử ở `v`, còn `carry = total_here − 1` phần tử phải đẩy sang `v + 1`; mỗi lần sang ô mới tất cả `carry` phần tử tốn `1` nước: `moves += carry`.
3. Dừng khi `carry = 0` sau giá trị lớn nhất.

In [7]:
def increments_counting(nums):
    n = len(nums)
    top = max(nums)
    cnt = [0] * (top + n + 1)
    for x in nums:
        cnt[x] += 1
    moves = carry = 0
    for v in range(len(cnt)):
        here = cnt[v] + carry
        carry = max(here - 1, 0)                         # một phần tử ở lại v, phần còn lại bị đẩy sang v + 1
        moves += carry                                   # mỗi phần tử bị đẩy sang v + 1 tốn 1 nước
    return moves


for a, want in examples:
    assert increments_counting(a) == want
print([increments_counting(a) for a, _ in examples])

[1, 6]


In [8]:
# Code nộp LeetCode cho Cách 2 (O(n + max))
class SolutionCounting:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        n, top = len(nums), max(nums)
        cnt = [0] * (top + n + 1)
        for x in nums:
            cnt[x] += 1
        moves = carry = 0
        for v in range(len(cnt)):
            here = cnt[v] + carry
            carry = max(here - 1, 0)
            moves += carry
        return moves

**Độ phức tạp:** `O(n + max)` thời gian (tuyến tính, không sắp xếp), `O(n + max)` bộ nhớ.

### Cách 3 — Union-Find "ô trống kế tiếp"

**Ý tưởng:** `find(v)` trả ô trống nhỏ nhất `≥ v`. Với mỗi phần tử `x` (thứ tự bất kỳ): `t = find(x)`; chiếm ô `t` bằng `parent[t] = t + 1`; chi phí `t − x`. Vì việc chọn ô trống nhỏ nhất `≥ x` cho từng phần tử, **không phụ thuộc thứ tự xử lý**, tổng chi phí không đổi (xem mục 10).

In [9]:
def increments_dsu(nums):
    parent = {}

    def find(v):                                         # ô trống nhỏ nhất >= v, kèm nén đường đi (lặp)
        root = v
        while root in parent:
            root = parent[root]
        while v in parent and parent[v] != root:
            parent[v], v = root, parent[v]
        return root

    moves = 0
    for x in nums:                                       # thứ tự bất kỳ
        t = find(x)
        parent[t] = t + 1                                # ô t đã bị chiếm: nhảy tới ô kế tiếp
        moves += t - x
    return moves


for a, want in examples:
    assert increments_dsu(a) == want
print([increments_dsu(a) for a, _ in examples])

[1, 6]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionDSU:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        parent = {}

        def find(v):
            root = v
            while root in parent:
                root = parent[root]
            while v in parent and parent[v] != root:
                parent[v], v = root, parent[v]
            return root

        moves = 0
        for x in nums:
            t = find(x)
            parent[t] = t + 1
            moves += t - x
        return moves

**Độ phức tạp:** `O(n α(n))` gộp (nén đường đi), bộ nhớ `O(n)`. Không cần sắp xếp, tiện cho dữ liệu **trực tuyến**.

### Cách 4 — Điền các khe trống bằng hai con trỏ

**Ý tưởng:** tách `nums` đã sắp xếp thành **giữ lại** (lần xuất hiện đầu của mỗi giá trị) và **dư** (các lần lặp lại). Các giá trị dư (đã sắp xếp) lần lượt chiếm các **khe trống** (số nguyên không thuộc tập giữ lại) nhỏ nhất `≥` giá trị của nó, duyệt bằng một con trỏ khe tăng dần.

In [11]:
def increments_gaps(nums):
    xs = sorted(nums)
    kept, extra = [], []
    for i, x in enumerate(xs):
        (extra if i and xs[i - 1] == x else kept).append(x)      # lần lặp lại thứ 2 trở đi là phần tử dư
    used = set(kept)
    slot = 0                                              # con trỏ khe trống, chỉ tăng
    moves = 0
    for x in extra:                                       # extra đã tăng dần
        slot = max(slot, x)                               # khe không nhỏ hơn x
        while slot in used:
            slot += 1
        used.add(slot)                                    # chiếm khe
        moves += slot - x
        slot += 1
    return moves


for a, want in examples:
    assert increments_gaps(a) == want
print([increments_gaps(a) for a, _ in examples])

[1, 6]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionGaps:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        nums.sort()
        kept, extra = [], []
        for i, x in enumerate(nums):
            (extra if i and nums[i - 1] == x else kept).append(x)
        used, slot, moves = set(kept), 0, 0
        for x in extra:
            slot = max(slot, x)
            while slot in used:
                slot += 1
            used.add(slot)
            moves += slot - x
            slot += 1
        return moves

**Độ phức tạp:** `O(n log n)` (sắp xếp) + `O(n + khoảng giá trị)` cho con trỏ khe, bộ nhớ `O(n)`.

## 8. TRACE từng bước

**Trace Cách 2 (đếm)** với `nums = [3,2,1,2,1,7]`: `cnt[1] = 2`, `cnt[2] = 2`, `cnt[3] = 1`, `cnt[7] = 1`.

| `v` | `cnt[v]` | `carry` trước | `here` | `carry` sau | `moves` cộng |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 0 |
| 1 | 2 | 0 | 2 | 1 | 1 |
| 2 | 2 | 1 | 3 | 2 | 2 |
| 3 | 1 | 2 | 3 | 2 | 2 |
| 4 | 0 | 2 | 2 | 1 | 1 |
| 5 | 0 | 1 | 1 | 0 | 0 |
| 6 | 0 | 0 | 0 | 0 | 0 |
| 7 | 1 | 0 | 1 | 0 | 0 |

Tổng `1 + 2 + 2 + 1 = 6` ✓ (phần tử `v = 1`: một phần tử bị đẩy lên `2` tốn `1`; hai phần tử bị đẩy lên `3` tốn `2`; ... khớp bảng tham lam). Hàm trace:

In [13]:
def trace_counting(nums):
    n, top = len(nums), max(nums)
    cnt = [0] * (top + n + 1)
    for x in nums:
        cnt[x] += 1
    moves = carry = 0
    print(f"{'v':>3} {'cnt':>4} {'carry trước':>12} {'here':>5} {'carry sau':>10} {'moves':>6}")
    for v in range(top + 1):
        before = carry
        here = cnt[v] + carry
        carry = max(here - 1, 0)
        moves += carry
        print(f"{v:>3} {cnt[v]:>4} {before:>12} {here:>5} {carry:>10} {moves:>6}")
    v = top + 1
    while carry:                                         # còn dư sau giá trị lớn nhất: dồn tiếp
        before = carry
        here = carry
        carry = here - 1
        moves += carry
        print(f"{v:>3} {0:>4} {before:>12} {here:>5} {carry:>10} {moves:>6}")
        v += 1
    print("=> số nước =", moves)
    return moves


assert trace_counting([3, 2, 1, 2, 1, 7]) == 6

  v  cnt  carry trước  here  carry sau  moves
  0    0            0     0          0      0
  1    2            0     2          1      1
  2    2            1     3          2      3
  3    1            2     3          2      5
  4    0            2     2          1      6
  5    0            1     1          0      6
  6    0            0     0          0      6
  7    1            0     1          0      6
=> số nước = 6


**Trace Cách 3 (Union-Find)** với `[1, 1, 2]`: `x = 1`: `find(1) = 1`, chiếm `parent[1] = 2`, chi phí `0`. `x = 1`: `find(1)`: `1 → 2` (trống) ⇒ `2`, chiếm `parent[2] = 3`, chi phí `1`. `x = 2`: `find(2)`: `2 → 3` ⇒ `3`, chi phí `1`. Tổng `2` (khớp `[1,2,3]`).

## 9. Code nộp LeetCode

Chọn **Cách 1** (sắp xếp + tham lam, ngắn và chắc). Giữ nguyên `class Solution`, `minIncrementForUnique(self, nums)`:

In [14]:
class Solution:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        nums = sorted(nums)
        t, moves = -1, 0                                   # t: đích lớn nhất đã dùng
        for x in nums:
            t = max(t + 1, x)                              # ô trống nhỏ nhất không nhỏ hơn x và lớn hơn mọi đích trước
            moves += t - x                                 # mỗi đơn vị tăng là một nước
        return moves


for a, want in examples:
    assert Solution().minIncrementForUnique(list(a)) == want
print([Solution().minIncrementForUnique(list(a)) for a, _ in examples])

[1, 6]


## 10. Vì sao đúng

**(a) Quy về gán đích.** Mọi lời giải ứng với một dãy đích phân biệt `t_i ≥ x_i`; số nước `= Σ t_i − Σ x_i`. (Tăng từng bước một tới `t_i` luôn thực hiện được.) Ta tối thiểu hoá `Σ t_i`.

**(b) Dạng chuẩn.** Cho `x₁ ≤ … ≤ x_n` và một gán đích tối ưu bất kỳ. Nếu có `i < j` (nên `x_i ≤ x_j`) với `t_i > t_j` thì hoán đổi hai đích: phần tử `i` nhận `t_j`, mà `t_j ≥ x_j ≥ x_i` nên hợp lệ; phần tử `j` nhận `t_i > t_j ≥ x_j` nên hợp lệ; tổng đích không đổi. Lặp lại tới khi đích **tăng chặt** theo thứ tự đã sắp (các đích phân biệt nên không có dấu bằng). Vậy tồn tại nghiệm tối ưu với `t_1 < t_2 < … < t_n`.

**(c) Tham lam tối ưu.** Với đích tăng chặt: `t_1 ≥ x_1`, `t_i ≥ max(x_i, t_{i−1} + 1)`. Quy nạp: `t_i* = max(x_i, t*_{i−1} + 1)` là cận dưới của mọi nghiệm tăng chặt (vì `t_{i−1} ≥ t*_{i−1}` theo quy nạp), và chính nó khả thi. Do đó nghiệm tham lam đạt cực tiểu từng `t_i`, nên cũng cực tiểu `Σ t_i`.

**(d) Cách 2 = Cách 1.** Duyệt giá trị `v` tăng dần: các đích của tham lam nằm đúng ở các `v` mà tại đó có phần tử ở lại. Tại `v`, `here = cnt[v] + carry` phần tử muốn ở `v`; đúng một phần tử chiếm `v` (đích tham lam là tăng chặt, mỗi `v` nhiều nhất một), còn `here − 1` dồn lên `v + 1`; mỗi phần tử dồn thêm một bậc tốn `1` nước nên `moves += here − 1`. Tổng các `moves` bằng `Σ (t_i − x_i)`.

**(e) Cách 3.** Cách 3 là **dò tuyến tính (linear probing)**: mỗi phần tử đặt vào ô trống nhỏ nhất `≥ x`. Tính chất quen thuộc của dò tuyến tính: **tập ô bị chiếm cuối cùng không phụ thuộc thứ tự chèn** (mỗi ô `s` bị chiếm khi và chỉ khi, với một khoảng `[a, s]` nào đó, số phần tử có giá trị trong `[a, s]` lớn hơn hoặc bằng số ô của khoảng — điều kiện chỉ phụ thuộc đa tập giá trị). Do đó `Σ t` bằng nhau với mọi thứ tự, kể cả thứ tự đã sắp xếp (Cách 1), và `Σ t − Σ x` là đáp án. Ta kiểm chứng bằng máy ở mục 12 (thứ tự bất kỳ cho cùng đáp án).

**(f) Cách 4.** Các phần tử giữ lại (lần đầu của mỗi giá trị) chiếm chính giá trị của mình (chi phí `0`); phần tử dư tiến lên khe trống kế tiếp theo thứ tự tăng dần, giống như tham lam.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **vét cạn tập ô đích** trên mọi mảng nhỏ (mục 12D), và oracle này lại được đối chiếu với **hoán vị đầy đủ** (12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp + tham lam | `O(n log n)` | `O(1)`–`O(n)` | **đáp án chính** |
| 2. Mảng đếm + mang số dư | `O(n + max)` | `O(n + max)` | tuyến tính, không sắp xếp |
| 3. Union-Find ô trống | `O(n α(n))` | `O(n)` | trực tuyến, thứ tự bất kỳ |
| 4. Điền khe bằng hai con trỏ | `O(n log n)` | `O(n)` | rõ ý "giữ / dư" |

Đo thực tế (`n = 10^5`, trường hợp xấu: mọi phần tử bằng nhau):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(list(arg))
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(945)
datasets = {
    "ngẫu nhiên": [rng.randint(0, 10 ** 5) for _ in range(10 ** 5)],
    "toàn bằng nhau": [10 ** 5] * 10 ** 5,
    "toàn 0": [0] * 10 ** 5,
}
for label, a in datasets.items():
    row = [f"{name} {bench(fn, a) * 1e3:8.1f} ms" for name, fn in
           [("greedy", increments_greedy), ("counting", increments_counting), ("dsu", increments_dsu), ("gaps", increments_gaps)]]
    print(f"{label:16s}: " + " | ".join(row))

ngẫu nhiên      : greedy     28.8 ms | counting     21.2 ms | dsu     76.4 ms | gaps     44.0 ms


toàn bằng nhau  : greedy      8.5 ms | counting     19.7 ms | dsu     38.1 ms | gaps     20.2 ms


toàn 0          : greedy      7.6 ms | counting     13.0 ms | dsu     36.3 ms | gaps     24.1 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle tập ô đích (kiểm oracle bằng hoán vị đầy đủ), (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(nums):
    """Vét cạn: chọn TẬP n ô đích phân biệt trong [0, max + n - 1]; hợp lệ nếu ghép sắp-với-sắp thoả slot_i >= x_i (tối ưu theo
    bất đẳng thức hoán vị); chi phí = tổng ô - tổng x. Lấy nhỏ nhất."""
    n = len(nums)
    xs = sorted(nums)
    top = max(nums) + n - 1
    best = None
    for slots in itertools.combinations(range(top + 1), n):         # slots đã tăng dần
        if all(s >= x for s, x in zip(slots, xs)):
            cost = sum(slots) - sum(xs)
            best = cost if best is None else min(best, cost)
    return best


def oracle_permutations(nums):
    """Vét cạn HOÀN TOÀN: mọi cách gán đích phân biệt cho từng phần tử (hoán vị các ô), không dựa vào bất đẳng thức hoán vị."""
    n = len(nums)
    top = max(nums) + n - 1
    best = None
    for slots in itertools.combinations(range(top + 1), n):
        for perm in itertools.permutations(slots):
            if all(t >= x for t, x in zip(perm, nums)):
                cost = sum(perm) - sum(nums)
                best = cost if best is None else min(best, cost)
    return best


IMPLS = {
    "greedy": increments_greedy,
    "counting": increments_counting,
    "dsu": increments_dsu,
    "gaps": increments_gaps,
    "Solution": lambda a: Solution().minIncrementForUnique(list(a)),
    "SolutionGreedy": lambda a: SolutionGreedy().minIncrementForUnique(list(a)),
    "SolutionCounting": lambda a: SolutionCounting().minIncrementForUnique(list(a)),
    "SolutionDSU": lambda a: SolutionDSU().minIncrementForUnique(list(a)),
    "SolutionGaps": lambda a: SolutionGaps().minIncrementForUnique(list(a)),
}


def check(nums, want=None, names=None):
    if want is None:
        want = oracle(nums)
    for name in (names or IMPLS):
        got = IMPLS[name](list(nums))
        assert got == want, (name, nums, got, want)


for a, want in examples:
    assert oracle(a) == oracle_permutations(a) == want
    check(a, want)
print("ví dụ đề: oracle tập ô đích = oracle hoán vị đầy đủ và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle tập ô đích = oracle hoán vị đầy đủ và 9 cài đặt đúng


### B. Edge case

- Một phần tử: `0`.
- Mọi phần tử khác nhau: `0`.
- Mọi phần tử bằng nhau (`n` lần `v`): `n(n−1)/2`.
- Hai phần tử bằng nhau: `1`.
- Giá trị lớn nhất `10^5` lặp nhiều lần (không bị chặn trên `10^5`).
- Giá trị `0` lặp nhiều lần.
- Mảng đã sắp xếp / ngược / ngẫu nhiên với trùng xen kẽ.
- Khe trống ngay giữa hai nhóm trùng (đích nhảy qua giá trị đã có).

In [17]:
cases = [
    ([5], 0), ([1, 2, 3], 0), ([7, 7], 1), ([0, 0, 0], 3), ([4, 4, 4, 4], 6), ([2, 2, 3], 2),
    ([1, 1, 1, 3, 3], 6), ([3, 3, 3, 3, 4], 9), ([10 ** 5] * 5, 10), ([0] * 6, 15), ([5, 3, 3, 5, 4], None),
]
for a, want in cases:
    check(a, want)
assert Solution().minIncrementForUnique([10 ** 5] * 100) == 100 * 99 // 2
print(len(cases), "edge case +1: đúng")

11 edge case +1: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_no_sort(nums):                                 # không sắp xếp
    t, moves = -1, 0
    for x in nums:
        t = max(t + 1, x)
        moves += t - x
    return moves


def bug_count_duplicates(nums):                        # mỗi phần tử trùng chỉ +1
    return len(nums) - len(set(nums))


def bug_no_max(nums):                                  # y + 1 mà không max với x (có thể làm giảm giá trị: kết quả âm)
    t, moves = -1, 0
    for x in sorted(nums):
        t = t + 1
        moves += t - x
    return moves


def bug_init_zero(nums):                               # khởi tạo t = 0 (thay vì -1): đẩy phần tử 0 đầu tiên lên 1
    t, moves = 0, 0
    for x in sorted(nums):
        t = max(t + 1, x)
        moves += t - x
    return moves


def bug_decrement(nums):                               # cho phép giảm: chuẩn hoá về 0..n-1 (sai luật)
    return sum(abs(x - i) for i, x in enumerate(sorted(nums)))


def bug_counting_no_carry(nums):                       # mảng đếm nhưng không dồn số dư (chỉ đẩy một bước)
    cnt = {}
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
    return sum(c - 1 for c in cnt.values())


BUGS = [bug_no_sort, bug_count_duplicates, bug_no_max, bug_init_zero, bug_decrement, bug_counting_no_carry]
rng = random.Random(3)
tests = [[rng.randint(0, 4) for _ in range(rng.randint(1, 6))] for _ in range(1000)]
for bug in BUGS:
    bad = next((a for a in tests if bug(a) != oracle(a)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên {bad}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_no_sort            sai trên [4, 0, 4, 0]: ra 14, đúng 2
bug_count_duplicates   sai trên [2, 4, 1, 1]: ra 1, đúng 2
bug_no_max             sai trên [4, 4]: ra -7, đúng 1
bug_init_zero          sai trên [4, 0, 4, 0]: ra 4, đúng 2
bug_decrement          sai trên [4, 4]: ra 7, đúng 1
bug_counting_no_carry  sai trên [2, 4, 1, 1]: ra 1, đúng 2


### D. Vét cạn mọi mảng nhỏ

(1) **Kiểm oracle:** `oracle` (tập ô đích) khớp `oracle_permutations` (hoán vị đầy đủ) trên mọi mảng độ dài `1..4` với giá trị `0..3` (`340` mảng). (2) **Kiểm cài đặt:** mọi mảng độ dài `1..6` với giá trị `0..3` (`4 + 16 + 64 + 256 + 1 024 + 4 096 = 5 460`) và độ dài `7` với giá trị `0..2` (`2 187`) khớp `oracle`.

In [19]:
t0 = time.perf_counter()
cnt = 0
for n in range(1, 5):
    for tup in itertools.product(range(4), repeat=n):
        assert oracle(list(tup)) == oracle_permutations(list(tup)), tup
        cnt += 1
print(cnt, "mảng nhỏ: oracle tập ô đích = oracle hoán vị đầy đủ")
cnt2 = 0
fast = ("greedy", "counting", "dsu", "gaps", "Solution")
for n, hi in [(1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 3)]:
    for tup in itertools.product(range(hi), repeat=n):
        a = list(tup)
        want = oracle(a)
        for name in fast:
            assert IMPLS[name](a) == want, (name, a)
        cnt2 += 1
print(cnt2, "mảng: 5 cài đặt khớp oracle;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 340 and cnt2 == 5460 + 2187

340 mảng nhỏ: oracle tập ô đích = oracle hoán vị đầy đủ


7647 mảng: 5 cài đặt khớp oracle; 0.7 s


### E. Ngẫu nhiên có seed và tính chất

Mảng ngẫu nhiên tới `300` phần tử, giá trị `0..100` hoặc `0..10^5`. Tính chất: (1) hoán vị mảng không đổi đáp án; (2) đáp án `≥ n − (số giá trị khác nhau)`; (3) đáp án `= 0` ⇔ các phần tử đã phân biệt; (4) **cộng cùng một hằng số** vào mọi phần tử không đổi đáp án; (5) sau khi áp dụng đích tham lam, các giá trị **phân biệt** và mỗi đích `≥` phần tử gốc.

In [20]:
rng = random.Random(20240945)
for _ in range(1500):
    n = rng.randint(1, 300)
    hi = rng.choice([5, 100, 10 ** 5])
    a = [rng.randint(0, hi) for _ in range(n)]
    want = increments_greedy(a)
    for name in ("counting", "dsu", "gaps", "Solution", "SolutionCounting", "SolutionDSU", "SolutionGaps"):
        assert IMPLS[name](a) == want, (name, a)
    b = a[:]
    rng.shuffle(b)
    assert Solution().minIncrementForUnique(b) == want                                    # (1)
    assert want >= n - len(set(a))                                                        # (2)
    assert (want == 0) == (len(set(a)) == n)                                              # (3)
    c = rng.randint(1, 50)
    assert Solution().minIncrementForUnique([x + c for x in a]) == want                   # (4)
    xs, ts = greedy_targets(a)
    assert len(set(ts)) == n and all(t >= x for t, x in zip(ts, xs)) and sum(ts) - sum(xs) == want   # (5)
for _ in range(200):
    a = [rng.randint(0, 6) for _ in range(rng.randint(1, 7))]
    assert Solution().minIncrementForUnique(a) == oracle(a)
print("1500 mảng ngẫu nhiên: 8 cài đặt khớp; 5 tính chất (hoán vị, cận dưới, điều kiện 0, tịnh tiến, đích phân biệt) và 200 mảng nhỏ khớp oracle")

1500 mảng ngẫu nhiên: 8 cài đặt khớp; 5 tính chất (hoán vị, cận dưới, điều kiện 0, tịnh tiến, đích phân biệt) và 200 mảng nhỏ khớp oracle


### F. Giới hạn: `n = 10^5`

Các cách khớp nhau trên mảng lớn; công thức đóng cho các dạng đặc biệt (toàn bằng nhau, mọi phần tử khác nhau, hai nhóm), và kiểm tra tràn `int` 32-bit.

In [21]:
rng = random.Random(500)
n = 10 ** 5
fast = ["greedy", "counting", "dsu", "gaps", "Solution", "SolutionGreedy", "SolutionCounting", "SolutionDSU", "SolutionGaps"]
datasets = {
    "ngẫu nhiên 0..10^5": [rng.randint(0, 10 ** 5) for _ in range(n)],
    "ngẫu nhiên 0..100": [rng.randint(0, 100) for _ in range(n)],
    "toàn 10^5": [10 ** 5] * n,
    "toàn 0": [0] * n,
    "mọi giá trị khác nhau": list(range(n)),
    "hai nhóm": [0] * 50000 + [10 ** 5] * 50000,
}
for name, a in datasets.items():
    res = {k: IMPLS[k](a) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    print(f"{name:24s}: {res['greedy']}")
assert Solution().minIncrementForUnique([10 ** 5] * n) == n * (n - 1) // 2 > 2 ** 31 - 1
assert Solution().minIncrementForUnique([0] * n) == n * (n - 1) // 2
assert Solution().minIncrementForUnique(list(range(n))) == 0
assert Solution().minIncrementForUnique([0] * 50000 + [10 ** 5] * 50000) == 2 * (50000 * 49999 // 2)
print("công thức đóng khớp; kết quả toàn bằng nhau vượt 2^31 - 1: cần long ở ngôn ngữ 32-bit")

ngẫu nhiên 0..10^5      : 14035408


ngẫu nhiên 0..100       : 4994954387


toàn 10^5               : 4999950000


toàn 0                  : 4999950000
mọi giá trị khác nhau   : 0


hai nhóm                : 2499950000
công thức đóng khớp; kết quả toàn bằng nhau vượt 2^31 - 1: cần long ở ngôn ngữ 32-bit


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không sắp xếp

In [22]:
print("[3,2,1,2,1,7]:", bug_no_sort([3, 2, 1, 2, 1, 7]), "| đúng:", Solution().minIncrementForUnique([3, 2, 1, 2, 1, 7]))
assert bug_no_sort([3, 2, 1, 2, 1, 7]) != 6

[3,2,1,2,1,7]: 17 | đúng: 6


**Vì sao sai:** tham lam `max(t + 1, x)` chỉ đúng khi duyệt tăng dần. **Đúng:** sắp xếp trước (hoặc dùng đếm/Union-Find).

### Lỗi 2: chỉ đếm số phần tử trùng

In [23]:
print("[1,1,1]:", bug_count_duplicates([1, 1, 1]), "| đúng:", Solution().minIncrementForUnique([1, 1, 1]))
assert bug_count_duplicates([1, 1, 1]) == 2 and Solution().minIncrementForUnique([1, 1, 1]) == 3

[1,1,1]: 2 | đúng: 3


**Vì sao sai:** mỗi phần tử dư có thể phải tăng **nhiều** bước (`1, 2, …`). **Đúng:** cộng `t − x` thực tế.

### Lỗi 3: bỏ `max(·, x)`

In [24]:
print("[1,5]:", bug_no_max([1, 5]), "| đúng:", Solution().minIncrementForUnique([1, 5]))
assert bug_no_max([1, 5]) < 0 and Solution().minIncrementForUnique([1, 5]) == 0

[1,5]: -5 | đúng: 0


**Vì sao sai:** đích không được **nhỏ hơn** giá trị gốc (chỉ được tăng). **Đúng:** `t = max(t + 1, x)`.

### Lỗi 4: khởi tạo `t = 0`

In [25]:
print("[0,5]:", bug_init_zero([0, 5]), "| đúng:", Solution().minIncrementForUnique([0, 5]))
assert bug_init_zero([0, 5]) == 1 and Solution().minIncrementForUnique([0, 5]) == 0

[0,5]: 1 | đúng: 0


**Vì sao sai:** `t` là đích đã dùng; trước phần tử đầu chưa dùng gì nên `t = -1` (để `t + 1 = 0` hợp lệ). **Đúng:** `t = -1` (hoặc `-∞`).

### Lỗi 5: cho phép giảm giá trị

In [26]:
print("[5,5,5]:", bug_decrement([5, 5, 5]), "| đúng:", Solution().minIncrementForUnique([5, 5, 5]))
assert bug_decrement([5, 5, 5]) != Solution().minIncrementForUnique([5, 5, 5])

[5,5,5]: 12 | đúng: 3


**Vì sao sai:** chuẩn hoá về `0..n−1` yêu cầu giảm. Đề chỉ cho **tăng**. **Đúng:** đích không nhỏ hơn giá trị gốc.

### Lỗi 6 (bonus): tràn `int` 32-bit

In [27]:
total = Solution().minIncrementForUnique([10 ** 5] * 10 ** 5)
wrapped = ((total + 2 ** 31) % 2 ** 32) - 2 ** 31
print("đúng:", total, "| mô phỏng int32:", wrapped)
assert wrapped != total

đúng: 4999950000 | mô phỏng int32: 704982704


**Vì sao sai:** ở Java/C++ dùng `int` sẽ tràn. **Đúng:** dùng `long` cho biến tích luỹ (đề đảm bảo test thật vừa 32-bit nhưng nên dùng `long` để an toàn).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "làm các giá trị phân biệt bằng thao tác tăng/giảm với chi phí tối thiểu" → **sắp xếp + tham lam** (đích tăng chặt), hoặc **đếm + mang số dư**, hoặc **Union-Find ô trống kế tiếp**.

**Mẹo:**

1. Quy về `Σ đích − Σ gốc`.
2. Tham lam từ nhỏ đến lớn với `t = max(t + 1, x)`.
3. Khi giá trị nhỏ, đếm bằng mảng (không sắp xếp).
4. Union-Find "ô trống kế tiếp" cho dữ liệu trực tuyến.
5. Dùng `long` cho tổng.
6. Oracle: chọn tập ô đích bằng vét cạn (và kiểm oracle bằng hoán vị).

**Họ bài cùng pattern:** Minimum Moves to Make Array Elements Equal (453), Smallest Missing Positive (41), Assign Cookies (455), Car Fleet (853), Sort Colors (75), Minimum Number of Operations to Make Array Continuous (2009), Sequentially Ordinal Rank Tracker.

## 15. Biến thể

1. **Cho phép cả tăng và giảm** (chi phí `|t − x|`): bài toán khác (ghép sắp-với-sắp vào `0..n−1` hoặc ô tuỳ ý).
2. **Giá trị tối đa bị chặn** (đích `≤ M`): có thể vô nghiệm.
3. **Đếm số cách** (không tối thiểu).
4. **Trả về mảng cuối cùng** (một nghiệm tối ưu).
5. **Chi phí tăng `k` đơn vị mỗi nước.**

Cài biến thể 1 (tăng/giảm, đích phân biệt) bằng vét cạn nhỏ và biến thể 4 (mảng cuối cùng):

In [28]:
def unique_final_array(nums):
    """Biến thể 4: trả MỘT mảng đích tối ưu (giữ nguyên thứ tự chỉ số gốc)."""
    order = sorted(range(len(nums)), key=lambda i: nums[i])
    out = [0] * len(nums)
    t = -1
    for i in order:
        t = max(t + 1, nums[i])
        out[i] = t
    return out


def make_unique_up_down(nums):
    """Biến thể 1: cho phép tăng và giảm; đích >= 0; chi phí sum |t - x|. Sắp xếp rồi ghép với n đích phân biệt tối ưu (vét cạn tập đích)."""
    n = len(nums)
    xs = sorted(nums)
    top = max(nums) + n
    best = None
    for slots in itertools.combinations(range(0, top + 1), n):
        cost = sum(abs(s - x) for s, x in zip(slots, xs))
        best = cost if best is None else min(best, cost)
    return best


rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(0, 5) for _ in range(rng.randint(1, 6))]
    final = unique_final_array(a)
    assert len(set(final)) == len(a) and all(f >= x for f, x in zip(final, a)) and sum(final) - sum(a) == Solution().minIncrementForUnique(a)
    assert make_unique_up_down(a) <= Solution().minIncrementForUnique(a)                  # được giảm thì không tệ hơn
print("mảng cuối cùng tối ưu hợp lệ; cho phép giảm cho chi phí không lớn hơn (500 mảng)")
assert make_unique_up_down([5, 5, 5]) == 2 and Solution().minIncrementForUnique([5, 5, 5]) == 3

mảng cuối cùng tối ưu hợp lệ; cho phép giảm cho chi phí không lớn hơn (500 mảng)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp + tham lam | `t = max(t + 1, x)` | `O(n log n)` | `O(1)`–`O(n)` | **đáp án chính** |
| 2. Đếm + mang số dư | dồn trùng lên giá trị kế | `O(n + max)` | `O(n + max)` | giá trị nhỏ, không sắp xếp |
| 3. Union-Find ô trống | `find(x)` = ô trống nhỏ nhất `≥ x` | `O(n α(n))` | `O(n)` | trực tuyến |
| 4. Điền khe | giữ lần đầu, lần lặp điền khe | `O(n log n)` | `O(n)` | giải thích rõ |

Cả bốn khớp oracle tập ô đích (được đối chiếu hoán vị đầy đủ) trên mọi mảng nhỏ và khớp nhau ở `n = 10^5` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().minIncrementForUnique([1, 1]), Solution().minIncrementForUnique([2, 2, 2, 2]), Solution().minIncrementForUnique([1, 2, 2, 3, 3]), Solution().minIncrementForUnique([0, 4, 4, 4]))

1 6 4 3


### Bài 2 (tính tay)

`nums = [2, 2, 3, 3, 3, 8]`. (a) Sắp xếp và đích tham lam? (b) Chi phí từng phần tử? (c) Tổng? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện tham lam:

In [30]:
def greedy_todo(nums):
    t, moves = -1, 0
    for x in sorted(nums):
        pass            # TODO 1: t = ô trống nhỏ nhất không nhỏ hơn x và lớn hơn t
        pass            # TODO 2: moves += t - x
    return moves


# Khi làm xong, bỏ comment để kiểm tra:
# assert [greedy_todo(a) for a, _ in examples] == [1, 6]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mảng đếm + mang số dư**:

In [31]:
class MySolution:
    def minIncrementForUnique(self, nums):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().minIncrementForUnique(list(a)) == w for a, w in examples)

### Bài 5 (mini challenge): Minimum Moves to Make Array Elements Equal (453)

Mỗi nước **tăng `n − 1` phần tử thêm `1`**. Số nước tối thiểu để mọi phần tử bằng nhau? (Gợi ý: tương đương **giảm một phần tử** đi `1`; đáp án `Σ x − n·min`.) Viết `min_moves_equal(nums)` và kiểm tra bằng mô phỏng BFS trên các mảng rất nhỏ.

In [32]:
def min_moves_equal(nums):
    pass            # TODO


# Khi làm xong:
# assert min_moves_equal([1, 2, 3]) == 3 and min_moves_equal([1, 1, 1]) == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: sắp xếp, đích tăng chặt.
- Mức 2: `[1,1]` → đích `1,2` ⇒ `1`; `[2,2,2,2]` → `2,3,4,5` ⇒ `0+1+2+3 = 6`.
- Mức 3: kết quả `1 6 4 3`: `[1,2,2,3,3]` → đích `1,2,3,4,5` ⇒ `0+0+1+1+2 = 4`; `[0,4,4,4]` → đích `0,4,5,6` ⇒ `0+0+1+2 = 3`.

**Bài 2**

- Mức 1: sắp xếp `[2,2,3,3,3,8]`.
- Mức 2: đích: `2, 3, 4, 5, 6, 8`.
- Mức 3: chi phí `0, 1, 1, 2, 3, 0` = `7`.

**Bài 3**

- Mức 1: `t = max(t + 1, x)`.
- Mức 2: `moves += t - x`.
- Mức 3: xem Cách 1.

**Bài 4**

- Mức 1: `cnt` dài `max + n + 1`.
- Mức 2: `here = cnt[v] + carry`; `carry = max(here - 1, 0)`.
- Mức 3: `moves += carry`.

**Bài 5**

- Mức 1: tăng `n − 1` phần tử = giảm một phần tử (so tương đối).
- Mức 2: mỗi phần tử phải giảm về `min`.
- Mức 3: `sum(nums) - len(nums) * min(nums)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().minIncrementForUnique([1, 1]), Solution().minIncrementForUnique([2, 2, 2, 2]), Solution().minIncrementForUnique([1, 2, 2, 3, 3]), Solution().minIncrementForUnique([0, 4, 4, 4]))
assert vals == (1, 6, 4, 3) == (oracle([1, 1]), oracle([2, 2, 2, 2]), oracle([1, 2, 2, 3, 3]), oracle([0, 4, 4, 4]))
print("Bài 1 OK:", vals)

Bài 1 OK: (1, 6, 4, 3)


### Bài 2

(a) đích `2, 3, 4, 5, 6, 8`. (b) chi phí `0, 1, 1, 2, 3, 0`. (c) `7`.

In [34]:
a = [2, 2, 3, 3, 3, 8]
xs, ts = greedy_targets(a)
assert ts == [2, 3, 4, 5, 6, 8] and [t - x for t, x in zip(ts, xs)] == [0, 1, 1, 2, 3, 0]
assert Solution().minIncrementForUnique(a) == oracle(a) == 7
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def greedy_answer(nums):
    t, moves = -1, 0
    for x in sorted(nums):
        t = max(t + 1, x)
        moves += t - x
    return moves


assert [greedy_answer(a) for a, _ in examples] == [1, 6]
for a in tests[:300]:
    assert greedy_answer(a) == oracle(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def minIncrementForUnique(self, nums):
        n, top = len(nums), max(nums)
        cnt = [0] * (top + n + 1)
        for x in nums:
            cnt[x] += 1
        moves = carry = 0
        for v in range(len(cnt)):
            carry = max(cnt[v] + carry - 1, 0)
            moves += carry
        return moves


assert all(MySolutionAnswer().minIncrementForUnique(list(a)) == w for a, w in examples)
for a in tests[:300]:
    assert MySolutionAnswer().minIncrementForUnique(list(a)) == oracle(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
from collections import deque


def min_moves_equal_answer(nums):
    return sum(nums) - len(nums) * min(nums)


def min_moves_equal_bfs(nums):
    start = tuple(sorted(nums))
    seen, q = {start}, deque([(start, 0)])
    while q:
        state, d = q.popleft()
        if len(set(state)) == 1:
            return d
        for i in range(len(state)):                       # tăng mọi phần tử trừ phần tử thứ i
            nxt = tuple(sorted(x + 1 if j != i else x for j, x in enumerate(state)))
            if nxt not in seen and max(nxt) <= 12:
                seen.add(nxt)
                q.append((nxt, d + 1))


assert min_moves_equal_answer([1, 2, 3]) == 3 and min_moves_equal_answer([1, 1, 1]) == 0
rng = random.Random(5)
for _ in range(100):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 4))]
    assert min_moves_equal_answer(a) == min_moves_equal_bfs(a), a
print("Bài 5 OK: khớp BFS trên mảng nhỏ")

Bài 5 OK: khớp BFS trên mảng nhỏ


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (làm giá trị phân biệt bằng thao tác tăng → sắp xếp + `t = max(t + 1, x)`):**

```python
class Solution:
    def minIncrementForUnique(self, nums: List[int]) -> int:
        nums.sort()
        t, moves = -1, 0
        for x in nums:
            t = max(t + 1, x)
            moves += t - x
        return moves
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp** (hoặc dùng đếm/Union-Find)?
- [ ] `t = max(t + 1, x)` (không để đích nhỏ hơn giá trị gốc)?
- [ ] `t` khởi tạo `-1`?
- [ ] Cộng `t − x` (không chỉ đếm số phần tử trùng)?
- [ ] Dùng `long` ở ngôn ngữ 32-bit (tổng có thể `~5·10^9`)?
- [ ] Đã thử `[1,2,2] → 1`, `[3,2,1,2,1,7] → 6`, một phần tử, toàn bằng nhau, mọi phần tử khác nhau?
- [ ] Đúng tên `class Solution`, `minIncrementForUnique(self, nums)`, không in debug.